# Spinodal Decomposition with Cahn-Hilliard

This notebook generates **randomised patterns reminiscent of spinodal decomposition** using the `cahn_hilliard` model — the N-dimensional Cahn-Hilliard equation.

When a homogeneous mixture is quenched into a thermodynamically unstable state, it spontaneously separates into two interpenetrating phases. The Cahn-Hilliard equation is the canonical *conserved* phase-field model for this process:

$$\frac{\partial u}{\partial t} = M\,\nabla^2\!\left(u^3 - u - \varepsilon^2 \nabla^2 u\right)$$

- $u$ — composition field (the two phases are $u \approx \pm 1$)
- $M$ — mobility
- $\varepsilon$ — interface width

Unlike Allen-Cahn (non-conserved), the **spatial mean of $u$ is conserved** — the volume fraction of each phase is fixed by the initial condition.

**Operator-learning task:** $u(\mathbf{x}, t{=}0) \rightarrow u(\mathbf{x}, t{=}T)$

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from pdeforge import generate_dataset, get_model, describe_model

## 1. The idea: randomness in, morphology out

The trick to generating *randomised* spinodal patterns: **you don't draw the patterns — the PDE does.**

Each sample starts from a near-uniform field at `mean_composition` plus tiny white noise:

$$u_0 = m + \eta\,\xi(\mathbf{x}), \qquad \eta \sim 0.02$$

This initial condition is *structureless* — just speckle. But the spinodal instability amplifies a band of wavenumbers around $k^\* = 1/(\sqrt{2}\,\varepsilon)$ exponentially, and the conserved nonlinear dynamics sharpen the result into the characteristic interconnected morphology.

- Every sample looks spinodal — the **physics** guarantees it.
- Every sample is distinct — the **random seed** of the initial condition guarantees it.

The model is self-describing:

In [ ]:
print(describe_model("cahn_hilliard"))

## 2. Generate a 2D dataset

The unified `generate_dataset` API — the same call used for every other PDEForge model. The spatial dimension is inferred from the `resolution` dict.

In [ ]:
dataset = generate_dataset(
    model="cahn_hilliard",
    n_samples=12,
    resolution={"x": 128, "y": 128},
    params={
        "epsilon": 0.01,          # interface width -> pattern length scale
        "mean_composition": 0.0,  # 0 -> balanced 50/50 labyrinth
        "time_end": 0.1,
    },
    seed=42,
)
print(dataset)

`inputs` are the noisy initial conditions, `outputs` the developed patterns — both shape `(n_samples, ny, nx)`.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6.4))
for i in range(6):
    axes[0, i].imshow(dataset.inputs[i], cmap="RdBu_r", vmin=-0.08, vmax=0.08)
    axes[0, i].set_title(f"u0  (sample {i})", fontsize=10)
    axes[0, i].axis("off")
    axes[1, i].imshow(dataset.outputs[i], cmap="RdBu_r", vmin=-1, vmax=1)
    axes[1, i].set_title(f"u_T  (sample {i})", fontsize=10)
    axes[1, i].axis("off")
fig.suptitle(
    "Structureless noise (top) -> distinct spinodal patterns (bottom)", fontsize=13
)
plt.tight_layout()
plt.show()

## 3. From noise to pattern: the instability at work

Call `solve(..., return_full=True)` on the model directly to watch a single sample evolve. The fastest-growing band emerges first; then the domains **coarsen** — Cahn-Hilliard coarsening grows the characteristic length scale as $\sim t^{1/3}$.

In [ ]:
model = get_model("cahn_hilliard")(
    resolution={"x": 128, "y": 128},
    time_end=0.2,
    _n_time_steps=6,
)
ic = model.generate_ic(seed=3)
trajectory = model.solve(ic, return_full=True)

times = np.linspace(0, 0.2, 6)
fig, axes = plt.subplots(1, 6, figsize=(20, 3.6))
for ax, frame, t in zip(axes, trajectory, times):
    ax.imshow(frame, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(f"t = {t:.3f}")
    ax.axis("off")
fig.suptitle("Instability growth and coarsening", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Morphology knob: `mean_composition`

Because the mean of $u$ is conserved, `mean_composition` $m$ fixes the volume fraction of each phase — and that sets the **morphology class**:

- $m \approx 0$ — bicontinuous **labyrinth** (both phases interconnected)
- $|m| \gtrsim 0.3$ — the minority phase pinches off into **droplets**

In [ ]:
compositions = [-0.4, -0.2, 0.0, 0.2, 0.4]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for ax, m in zip(axes, compositions):
    mdl = get_model("cahn_hilliard")(
        resolution={"x": 128, "y": 128}, mean_composition=m
    )
    sol = mdl.solve(mdl.generate_ic(seed=3))
    ax.imshow(sol, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(f"mean_composition = {m}")
    ax.axis("off")
fig.suptitle("Labyrinth (m=0) -> droplets (|m| large)", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Length-scale knob: `epsilon`

$\varepsilon$ sets the interface width and hence the characteristic pattern wavelength $\lambda^\* \approx 2\pi\sqrt{2}\,\varepsilon$. Smaller $\varepsilon$ gives finer, more intricate patterns — and needs more grid resolution to resolve the interfaces.

In [ ]:
epsilons = [0.005, 0.0075, 0.01, 0.015, 0.02]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for ax, eps in zip(axes, epsilons):
    mdl = get_model("cahn_hilliard")(resolution={"x": 128, "y": 128}, epsilon=eps)
    sol = mdl.solve(mdl.generate_ic(seed=3))
    ax.imshow(sol, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(f"epsilon = {eps}")
    ax.axis("off")
fig.suptitle("Smaller epsilon -> finer pattern", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Coarsening knob: `time_end`

Once the phases have separated, the interfaces keep evolving to reduce total interfacial energy — domains merge and coarsen. Sampling `time_end` across a dataset gives patterns at different coarsening stages.

In [ ]:
end_times = [0.02, 0.05, 0.1, 0.2, 0.4]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for ax, T in zip(axes, end_times):
    mdl = get_model("cahn_hilliard")(resolution={"x": 128, "y": 128}, time_end=T)
    sol = mdl.solve(mdl.generate_ic(seed=3))
    ax.imshow(sol, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(f"time_end = {T}")
    ax.axis("off")
fig.suptitle("Longer time -> coarser domains", fontsize=13)
plt.tight_layout()
plt.show()

## 7. It really is spinodal: the structure factor

A quantitative fingerprint of spinodal decomposition: the **structure factor** $S(k)$ — the radially-averaged power spectrum — has a single dominant peak. The system has *one* characteristic length scale. As the pattern coarsens, that peak marches toward lower $k$ (larger features).

In [ ]:
def structure_factor(field, n_bins=48):
    """Radially-averaged power spectrum of a 2D field."""
    f = np.abs(np.fft.fftn(field - field.mean())) ** 2
    n = field.shape[0]
    k = np.fft.fftfreq(n, d=1.0 / n) * 2 * np.pi
    KX, KY = np.meshgrid(k, k)
    kmag = np.sqrt(KX**2 + KY**2)
    bins = np.linspace(0, kmag.max(), n_bins + 1)
    which = np.clip(np.digitize(kmag.ravel(), bins) - 1, 0, n_bins - 1)
    sums = np.bincount(which, weights=f.ravel(), minlength=n_bins)
    counts = np.bincount(which, minlength=n_bins)
    kc = 0.5 * (bins[:-1] + bins[1:])
    return kc, sums / np.maximum(counts, 1)


mdl = get_model("cahn_hilliard")(
    resolution={"x": 128, "y": 128}, time_end=0.4, _n_time_steps=4
)
traj = mdl.solve(mdl.generate_ic(seed=3), return_full=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
for frame, t in list(zip(traj, np.linspace(0, 0.4, 4)))[1:]:
    kc, Sk = structure_factor(frame)
    ax1.plot(kc, Sk, label=f"t = {t:.2f}")
ax1.set_xlabel("wavenumber  k")
ax1.set_ylabel("S(k)")
ax1.set_xlim(0, 120)
ax1.set_title("Structure factor: single peak, drifts left as it coarsens")
ax1.legend()
ax2.imshow(traj[-1], cmap="RdBu_r", vmin=-1, vmax=1)
ax2.set_title("final pattern  (t = 0.4)")
ax2.axis("off")
plt.tight_layout()
plt.show()

## 8. Save the dataset

A `cahn_hilliard` dataset is a standard `PDEDataset` — split it for training, save and load it in any supported format.

In [ ]:
splits = dataset.split(train=0.6, val=0.15, cal=0.15, test=0.1, seed=0)
for name, ds in splits.items():
    print(f"{name}: {ds.n_samples} samples")

dataset.save("./spinodal_2d_data")
from pdeforge.io import load_dataset
loaded = load_dataset("./spinodal_2d_data")
print("reloaded:", loaded)

import shutil
shutil.rmtree("./spinodal_2d_data")

## 9. 3D spinodal decomposition

The **same model** generates 3D patterns — just add a `z` axis to the `resolution` dict. The solver is N-dimensional (`fftn`); nothing else changes.

In [ ]:
dataset_3d = generate_dataset(
    model="cahn_hilliard",
    n_samples=6,
    resolution={"x": 64, "y": 64, "z": 64},
    params={"epsilon": 0.015, "time_end": 0.08},
    seed=42,
)
print(dataset_3d)

Each 3D sample is a volume of shape `(nz, ny, nx)`. The quickest way to inspect a volume is three orthogonal centre-slices:

In [ ]:
vol = dataset_3d.outputs[0]
nz, ny, nx = vol.shape

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(vol[nz // 2, :, :], cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title(f"z-slice  (z = {nz // 2})")
axes[1].imshow(vol[:, ny // 2, :], cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title(f"y-slice  (y = {ny // 2})")
axes[2].imshow(vol[:, :, nx // 2], cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_title(f"x-slice  (x = {nx // 2})")
for ax in axes:
    ax.axis("off")
fig.suptitle("3D spinodal pattern - orthogonal centre-slices", fontsize=13)
plt.tight_layout()
plt.show()

For interactive exploration in Jupyter (with `ipywidgets` installed), `dataset_3d.visualize()` opens the dataset explorer with a sample slider plus slice-plane and slice-position controls:

```python
dataset_3d.visualize()
```

## 10. Interactive 3D rendering with PyVista

Orthogonal slices are useful, but the 3D spinodal *structure* — the bicontinuous interface winding through the volume — is best seen as an **isosurface**. The `u = 0` level set is exactly the dividing surface between the two phases.

[PyVista](https://pyvista.org) is the VTK-based renderer that FEniCS / dolfinx uses for 3D. The `cahn_hilliard` output is already a regular `(nz, ny, nx)` grid, so we wrap it in a `pyvista.ImageData` and extract the contour directly.

The slider scrubs across **realisations** — each is a different random seed, so a different bicontinuous structure, but all share the same spinodal length-scale statistics.

> **Rotatable view:** with only `pyvista` installed, each frame is a static inline image. For a fully rotatable / zoomable widget, install the interactive backend — `pip install trame trame-vtk trame-vuetify` — and re-run; the cell below auto-detects it.

In [ ]:
import pyvista as pv


def isosurface(volume, level=0.0):
    """u = level isosurface of a (nz, ny, nx) scalar volume, as a PyVista mesh."""
    nz, ny, nx = volume.shape
    grid = pv.ImageData(
        dimensions=(nx, ny, nz),
        spacing=(1.0 / nx, 1.0 / ny, 1.0 / nz),
    )
    grid.point_data["u"] = volume.ravel(order="C")
    return grid.contour([level], scalars="u")


# Interactive (rotatable) backend if trame is installed, else static images.
try:
    import trame  # noqa: F401

    pv.set_jupyter_backend("trame")
    print("PyVista backend: trame (rotatable / zoomable)")
except ImportError:
    pv.set_jupyter_backend("static")
    print(
        "PyVista backend: static images "
        "(pip install trame trame-vtk trame-vuetify for a rotatable view)"
    )

In [ ]:
from ipywidgets import IntSlider, interact


def show_realisation(realisation):
    surf = isosurface(dataset_3d.outputs[realisation], level=0.0)
    pl = pv.Plotter(window_size=(640, 560))
    pl.add_mesh(
        surf,
        scalars=surf.points[:, 2],  # colour by height — a depth cue
        cmap="RdBu_r",
        show_scalar_bar=False,
        smooth_shading=True,
    )
    pl.add_text(f"realisation {realisation}  (u = 0 isosurface)", font_size=10)
    pl.camera_position = "iso"
    pl.show()


interact(
    show_realisation,
    realisation=IntSlider(
        min=0, max=dataset_3d.n_samples - 1, value=0, description="realisation"
    ),
)

## 11. Binary masks

If your target is a **binary mask** rather than a continuous phase field, set `binarize=True`. The model thresholds the final field at `u = 0` and returns a hard `{0, 1}` two-phase mask. Everything else still applies — `mean_composition` sets the mask's fill fraction, `epsilon` its length scale.

The built-in viewers (`plot_sample_3d`, `dataset.visualize()`) detect binary fields and render them as a flat two-tone mask rather than a diverging colour map.

In [ ]:
mask_dataset = generate_dataset(
    model="cahn_hilliard",
    n_samples=8,
    resolution={"x": 128, "y": 128},
    params={"epsilon": 0.01, "mean_composition": 0.0, "binarize": True},
    seed=42,
)
print(mask_dataset)
print("unique output values:", np.unique(mask_dataset.outputs))
fills = mask_dataset.outputs.reshape(mask_dataset.n_samples, -1).mean(axis=1)
print("fill fractions:", fills.round(3))

fig, axes = plt.subplots(1, 6, figsize=(18, 3.2))
for ax, mask in zip(axes, mask_dataset.outputs):
    ax.imshow(mask, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle(
    "Binary spinodal masks - distinct realisations, same statistics", fontsize=13
)
plt.tight_layout()
plt.show()

`binarize=True` is exactly the continuous field thresholded at `u = 0` — same instability, same morphology, just a hard label instead of a smooth field:

In [ ]:
cont = generate_dataset(
    model="cahn_hilliard", n_samples=1, resolution={"x": 128, "y": 128},
    params={"epsilon": 0.01}, seed=42,
)
matches = np.array_equal(
    mask_dataset.outputs[0], (cont.outputs[0] > 0).astype(float)
)
print(f"binarize=True matches a manual 'outputs > 0' threshold: {matches}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4.3))
axes[0].imshow(cont.outputs[0], cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("continuous field  u_T")
axes[1].imshow(cont.outputs[0] > 0, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("binary mask  (u_T > 0)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Binary masks work in 3D too — the orthogonal-slice viewer auto-detects the mask and drops the diverging colour map:

In [ ]:
mask_3d = generate_dataset(
    model="cahn_hilliard",
    n_samples=6,
    resolution={"x": 64, "y": 64, "z": 64},
    params={"epsilon": 0.015, "binarize": True},
    seed=7,
)
print(mask_3d)

from pdeforge.visualization.interactive import plot_sample_3d

plot_sample_3d(
    mask_3d.inputs[0],
    mask_3d.outputs[0],
    input_names=mask_3d.input_names,
    output_names=mask_3d.output_names,
    title="3D binary mask - orthogonal slices (noise in, mask out)",
)
plt.show()

And the PyVista isosurface — for a `{0, 1}` mask the surface sits at `0.5`. The slider scrubs across binary-mask realisations:

In [ ]:
def show_mask_realisation(realisation):
    surf = isosurface(mask_3d.outputs[realisation], level=0.5)
    pl = pv.Plotter(window_size=(640, 560))
    pl.add_mesh(surf, color="lightgray", smooth_shading=True)
    pl.add_text(f"binary mask realisation {realisation}", font_size=10)
    pl.camera_position = "iso"
    pl.show()


interact(
    show_mask_realisation,
    realisation=IntSlider(
        min=0, max=mask_3d.n_samples - 1, value=0, description="realisation"
    ),
)

## Summary

The `cahn_hilliard` model generates randomised spinodal-decomposition patterns in 2D and 3D:

- **Randomness** enters through the white-noise initial condition (the `seed`); the **morphology** is produced by the spinodal instability — it is not drawn by hand.
- `mean_composition` sets the morphology (labyrinth ↔ droplets), `epsilon` the length scale, `time_end` the coarsening stage.
- The spatial mean of $u$ is conserved to machine precision.
- The same N-dimensional solver handles 2D and 3D — only the `resolution` dict changes.
- Set `binarize=True` to get hard `{0, 1}` two-phase masks instead of the continuous field.

For a diverse training set, keep `seed` varying per sample (the default) and optionally sample `mean_composition`, `epsilon`, and `time_end` from ranges to span morphologies, length scales, and coarsening stages.